## Import Library

In [25]:
import pandas as pd
import yaml

In [26]:
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [27]:
import os

raw_dir = config["data"]["processed_dir"]

modsec_file = os.path.join("..", raw_dir, "modsec_parsed.csv")
nginx_file = os.path.join("..", raw_dir, "nginx_parsed.csv")

nginx_df = pd.read_csv(nginx_file)
modsec_df = pd.read_csv(modsec_file)

In [28]:
print(nginx_df.columns)
print(modsec_df.columns)

Index(['ip', 'method', 'endpoint', 'status', 'endpoint_length', 'param_count',
       'has_query', 'attack', 'has_sql', 'has_script', 'has_path_traversal'],
      dtype='str')
Index(['ip', 'method', 'endpoint', 'status', 'endpoint_length', 'param_count',
       'has_query', 'attack', 'has_sql', 'has_script', 'has_path_traversal'],
      dtype='str')


In [29]:
nginx_df["req_key"] = (
    nginx_df["ip"].astype(str)
    + nginx_df["method"]
    + nginx_df["endpoint"]
    + nginx_df["param_count"].astype(str)
    + nginx_df["has_query"].astype(str)
)

modsec_df["req_key"] = (
    modsec_df["ip"].astype(str)
    + modsec_df["method"]
    + modsec_df["endpoint"]
    + modsec_df["param_count"].astype(str)
    + modsec_df["has_query"].astype(str)
)

nginx_filtered = nginx_df[
    ~nginx_df["req_key"].isin(modsec_df["req_key"])
]

dataset = pd.concat(
    [nginx_filtered, modsec_df],
    ignore_index=True
)

In [30]:
dataset = dataset.sample(frac=1, random_state=42)

In [31]:
dataset["attack"].value_counts()

attack
1    20008
0     1529
Name: count, dtype: int64

In [32]:
dataset.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)